# Knife-edge profile: derivative of `modo_semaforo_ish.txt`

This uses the same method as `../fundamental/derivada_perfil.ipynb`. The knife goes from covering
the beam to uncovering it, so the beam profile along the knife's travel is $I(t) = dP/dt$. The mode
should have about three lobes (the "traffic light" / semáforo shape).

**Caveat:** the knife was moved by hand, so its speed is neither known nor constant. The x axis is time,
and the positions and widths are only qualitative.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.special import erf

# PM100D log: 2 header lines, then "dd/mm/yyyy hh:mm:ss,fff \t value \t W" (decimal comma)
t, P = [], []
with open('modo_semaforo_ish.txt', encoding='latin-1') as fh:
    for line in fh.read().splitlines()[2:]:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        t.append(datetime.strptime(parts[0].strip(), '%d/%m/%Y %H:%M:%S,%f').timestamp())
        P.append(float(parts[1].replace(',', '.')))
t = np.array(t) - t[0]          # s
P = np.array(P) * 1e3           # mW
t_raw, P_raw = t.copy(), P.copy()
print(f'{len(t)} points')

## The "impossible" peak: the power meter switching range

The meter is on `Range Auto`. At about 4.7 mW (t ≈ 27.3 s) it switches range:
- one reading jumps +0.21 mW in 56 ms (about 12× the local slope),
- then nothing is logged for **0.38 s** while the meter switches,
- after that the noise is about 1.5× larger, because the higher range has coarser resolution.

That jump produces the spike in the derivative. To fix it, we drop the jumped reading (its timestamp is
wrong: it belongs after the dead time). There's also a mismatch across the gap, and it's **lost time, not a power
offset**: the mismatch divided by the local slope is about 0.2 s, here and in `weird_2_peak_mode`, even though
the slopes differ by 4×. So we shift the timestamps after the switch forward by that amount.
To avoid this next time, **fix the range manually** before scanning.

In [ ]:
gap = np.argmax(np.diff(t))                  # index right before the 0.38 s dead time
t_switch = t[gap]
print(f'dead time of {t[gap+1] - t[gap]:.3f} s at t = {t_switch:.2f} s')

bad = gap                                    # the jumped reading, logged just before the dead time
t, P = np.delete(t, bad), np.delete(P, bad)

before = (t > t_switch - 1) & (t < t_switch)
after = (t > t_switch) & (t < t_switch + 1.4)
tc = t_switch + 0.2
pb = np.polyfit(t[before], P[before], 1)
pa = np.polyfit(t[after], P[after], 1)
lost = (np.polyval(pa, tc) - np.polyval(pb, tc)) / ((pa[0] + pb[0]) / 2)
t[t > t_switch] += lost
print(f'time lost in the range switch: {lost:.3f} s (added back)')

# the beam is fully uncovered at ~54 s; the rest is just drift
t_max = 54
keep = t < t_max
t, P = t[keep], P[keep]

# second differences cancel the (locally linear) signal and leave the noise
for name, m in [('low range', t < t_switch), ('high range', t > t_switch)]:
    print(f'{name} noise ~ {np.std(np.diff(P[m], 2)) / np.sqrt(6) * 1e3:.1f} uW')

## Savitzky–Golay derivative

This mode is noisier than the fundamental (the hand shakes more over the longer scan, and the range is coarser), so it uses
a slightly wider window (1.5 s). The raw derivative is shown for comparison.

In [ ]:
dt = 0.02

def sg_deriv(tt, PP, win_s, order=3):
    tu = np.arange(tt[0], tt[-1], dt)
    n = int(round(win_s / dt)) | 1   # window length has to be odd
    return tu, savgol_filter(np.interp(tu, tt, PP), n, order, deriv=1, delta=dt)

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(t_raw, P_raw, '.', color='0.7', ms=2, label='raw')
ax[0].plot(t, P, 'k.', ms=2, label='corrected + cut')
ax[0].axvline(t_switch, color='r', ls=':', lw=1)
ax[0].set_ylabel('P [mW]')
ax[0].legend()
tu_raw, d_raw = sg_deriv(t_raw, P_raw, 1.0)
ax[1].plot(tu_raw, d_raw, color='0.7', lw=1, label='raw, SG 1 s')
for win in [1.0, 1.5, 2.5]:
    ax[1].plot(*sg_deriv(t, P, win), lw=1, label=f'corrected, SG {win} s')
ax[1].axvline(t_switch, color='r', ls=':', lw=1)
ax[1].set_xlabel('t [s]')
ax[1].set_ylabel('dP/dt [mW/s]')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

tu, dPdt = sg_deriv(t, P, 1.5)
print(f'area under dP/dt = {np.trapezoid(dPdt, tu):.3f} mW   vs   step = {P[-30:].mean() - P[:30].mean():.3f} mW')

## Three-Gaussian fit

We fit the sum of three Gaussians to $dP/dt$, and the matching sum of three erfs to $P$ itself.

In [ ]:
def gauss3(t, c, *p):
    return c + sum(a * np.exp(-2 * (t - t0)**2 / w**2) for a, t0, w in zip(p[0::3], p[1::3], p[2::3]))

def erf3(t, P0, *p):
    # area of a*exp(-2(t-t0)^2/w^2) is a*w*sqrt(pi/2)
    return P0 + sum(a * w * np.sqrt(np.pi / 2) / 2 * (1 + erf(np.sqrt(2) * (t - t0) / w))
                    for a, t0, w in zip(p[0::3], p[1::3], p[2::3]))

p0 = [0, 0.28, 18, 7, 0.42, 31, 5, 0.18, 44, 7]
pg, cg = curve_fit(lambda t, *p: gauss3(t, *p), tu, dPdt, p0=p0)
pe, ce = curve_fit(lambda t, *p: erf3(t, *p), t, P, p0=[1.1] + p0[1:])

for name, p, c in [('3 gauss on dP/dt', pg, cg), ('3 erf on P', pe, ce)]:
    err = np.sqrt(np.diag(c))
    print(name)
    for k in range(3):
        i = 1 + 3 * k
        print(f'  peak {k+1}: t0 = {p[i+1]:.2f} ± {err[i+1]:.2f} s,  w = {abs(p[i+2]):.2f} ± {err[i+2]:.2f} s,  '
              f'height = {p[i]:.3f} mW/s,  power = {p[i]*abs(p[i+2])*np.sqrt(np.pi/2):.2f} mW')

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(t, P, 'k.', ms=2, label='data')
ax[0].plot(t, erf3(t, *pe), 'r-', label='3 erf fit')
ax[0].set_ylabel('P [mW]')
ax[0].legend()
ax[1].plot(tu, dPdt, 'k-', lw=1, label='dP/dt (SG 1.5 s)')
ax[1].plot(tu, gauss3(tu, *pg), 'r-', label='3 gaussian fit')
for k in range(3):
    ax[1].plot(tu, gauss3(tu, pg[0], *pg[1 + 3*k: 4 + 3*k]), '--', lw=1)
ax[1].set_xlabel('t [s]')
ax[1].set_ylabel('dP/dt [mW/s]')
ax[1].legend()
plt.tight_layout()
plt.show()